# AI Engineer Challenge - Dot Indonesia
## Bagian 1A: Object Tracking Mobil dengan YOLO
## Bagian 2: Teori Artificial Intelligence

**Kandidat:** Gymnastiar Al K

Notebook ini berisi implementasi object tracking mobil menggunakan YOLO (You Only Look Once) 
serta jawaban teoritis seputar AI dan Machine Learning.

---
# Bagian 1A: Object Tracking Mobil dengan Pre-trained Model YOLO

Pipeline:
1. Install dan import dependencies
2. Load pre-trained YOLO model (YOLOv8)
3. Download video sample (lalu lintas jalan raya)
4. Implementasi deteksi + tracking mobil per frame
5. Visualisasi hasil dengan bounding boxes

In [ ]:
# @title 1. Install Dependencies
!pip install -q ultralytics opencv-python-headless matplotlib numpy tqdm

In [ ]:
# @title 2. Import Libraries
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, Video
from tqdm.notebook import tqdm
import os
import urllib.request
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"OpenCV: {cv2.__version__}")

## 3. Load Pre-trained YOLO Model

Kita gunakan **YOLOv8n** (nano) — versi paling ringan, cocok untuk Colab. 
YOLOv8 sudah pre-trained di dataset COCO yang memiliki class 'car', 'truck', 'bus', dll.

In [ ]:
# @title Load YOLOv8 Model
MODEL_NAME = 'yolov8n.pt'

model = YOLO(MODEL_NAME)
print(f"Model loaded: {MODEL_NAME}")

# COCO class IDs untuk kendaraan
VEHICLE_CLASSES = [2, 3, 5, 7]  # car, motorcycle, bus, truck
print(f"Vehicle labels: {[model.names[c] for c in VEHICLE_CLASSES]}")

## 4. Download Video Sample

Gunakan video lalu lintas dari repositori Ultralytics sebagai sample testing.

In [ ]:
# @title Download Sample Traffic Video
VIDEO_DIR = Path("videos")
VIDEO_DIR.mkdir(exist_ok=True)
VIDEO_PATH = str(VIDEO_DIR / "traffic.mp4")

# Download traffic video
!wget -q --show-progress -O {VIDEO_PATH} https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4

file_size = os.path.getsize(VIDEO_PATH) if os.path.exists(VIDEO_PATH) else 0
print(f"Video downloaded: {os.path.getsize(VIDEO_PATH)/1024:.1f} KB" if file_size > 1000 else "Download failed, check connection")

In [ ]:
# @title 5. Fungsi Deteksi & Tracking Kendaraan
def detect_and_track(frame, model, conf_threshold=0.4, vehicle_only=True):
    """
    Deteksi dan track kendaraan dalam frame menggunakan YOLO + ByteTrack.

    Args:
        frame: np.array (H, W, 3)
        model: YOLO model instance
        conf_threshold: minimum confidence score
        vehicle_only: filter hanya kendaraan (car, bus, truck, motorcycle)

    Returns:
        annotated_frame: frame dengan bounding boxes
        detections: list of dict
    """
    results = model.track(
        frame, persist=True, conf=conf_threshold, iou=0.5,
        tracker='bytetrack.yaml', verbose=False
    )

    detections = []
    annotated_frame = frame.copy()

    if results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        confs = results[0].boxes.conf.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
        track_ids = results[0].boxes.id.cpu().numpy().astype(int) if results[0].boxes.id is not None else [-1]*len(boxes)

        for box, conf, cls_id, track_id in zip(boxes, confs, cls_ids, track_ids):
            if vehicle_only and cls_id not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)
            label = model.names[cls_id]
            detections.append({'bbox': (x1, y1, x2, y2), 'confidence': float(conf),
                               'class': label, 'track_id': int(track_id)})

            # Warna per track ID
            color = tuple((track_id * 127 % 256, track_id * 63 % 256, track_id * 255 % 256)) if track_id > 0 else (0, 255, 0)

            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 3)
            track_str = f"ID:{track_id} " if track_id > 0 else ""
            label_text = f"{track_str}{label} {conf:.2f}"

            (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(annotated_frame, (x1, y1 - 25), (x1 + tw + 5, y1), color, -1)
            cv2.putText(annotated_frame, label_text, (x1 + 3, y1 - 7),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    return annotated_frame, detections

print("Fungsi detect_and_track siap digunakan")

In [ ]:
# @title 6. Proses Video Frame by Frame
def process_video(video_path, model, output_path='output_tracked.mp4',
                  conf_threshold=0.4, max_frames=None):
    """Proses seluruh video dengan deteksi + tracking per frame."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Tidak bisa membuka video: {video_path}")

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total_frames = min(total_frames, max_frames)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    frame_count = 0
    vehicle_counts = []

    print(f"Memproses {total_frames} frames | {fps} fps | {width}x{height}")
    with tqdm(total=total_frames, desc="Tracking") as pbar:
        while cap.isOpened() and frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            annotated_frame, detections = detect_and_track(frame, model, conf_threshold)
            cv2.putText(annotated_frame, f"Frame: {frame_count+1} | Vehicles: {len(detections)}",
                       (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            out.write(annotated_frame)
            vehicle_counts.append(len(detections))
            frame_count += 1
            pbar.update(1)

    cap.release(); out.release(); cv2.destroyAllWindows()
    print(f"\nVideo selesai: {output_path}")
    print(f"Rata-rata kendaraan/frame: {np.mean(vehicle_counts):.1f}")
    print(f"Max kendaraan dalam 1 frame: {np.max(vehicle_counts)}")
    return output_path, vehicle_counts

print("Fungsi process_video siap digunakan")

In [ ]:
# @title 7. Eksekusi Deteksi & Tracking
OUTPUT_VIDEO = 'output_tracked.mp4'
output_path, counts = process_video(VIDEO_PATH, model, OUTPUT_VIDEO, conf_threshold=0.4, max_frames=300)

# Kompres untuk tampil di Colab
COMPRESSED = 'output_compressed.mp4'
!ffmpeg -y -i {output_path} -vcodec libx264 -crf 23 {COMPRESSED} -loglevel error

In [ ]:
# @title 8. Tampilkan Hasil Video
if os.path.exists(COMPRESSED):
    display(Video(COMPRESSED, width=800))
elif os.path.exists(output_path):
    display(Video(output_path, width=800))
else:
    print("Video output tidak ditemukan")

In [ ]:
# @title 9. Sample Frame Visualisasi
def display_sample_frames(video_path, model, num_samples=6):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_indices = np.linspace(0, min(total-1, 300), num_samples, dtype=int)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, idx in enumerate(sample_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret: break
        annotated, dets = detect_and_track(frame, model, 0.4)
        axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"Frame {idx} | {len(dets)} kendaraan", fontsize=12)
        axes[i].axis('off')

    cap.release()
    plt.tight_layout()
    plt.show()

display_sample_frames(VIDEO_PATH, model, 6)

In [ ]:
# @title 10. Analisis Hasil
print("=" * 50)
print("ANALISIS HASIL DETEKSI & TRACKING")
print("=" * 50)
print(f"Model: {MODEL_NAME}")
print(f"Tracker: ByteTrack")
print(f"Kelas: car, bus, truck, motorcycle")
print(f"\nStatistik:")
print(f"  Total frame diproses: {len(counts)}")
print(f"  Rata-rata kendaraan/frame: {np.mean(counts):.1f}")
print(f"  Max kendaraan: {np.max(counts)}")
print(f"  Total deteksi: {np.sum(counts)}")

---
# Bagian 2: Teori Artificial Intelligence
---

## Soal 2a: Apa yang dimaksud dengan Artificial Intelligence (AI)?

**Artificial Intelligence (AI)** adalah cabang ilmu komputer yang berfokus pada
pembuatan sistem yang mampu melakukan tugas yang membutuhkan kecerdasan manusia.
AI bekerja dengan memproses data dalam jumlah besar, mengenali pola, dan membuat
keputusan atau prediksi berdasarkan pola tersebut.

**Dua contoh AI dalam kehidupan sehari-hari:**

1. **Rekomendasi Konten (YouTube, Netflix, Spotify)** — AI menganalisis riwayat
   tontonan/dengaran pengguna untuk merekomendasikan konten relevan menggunakan
   collaborative filtering dan deep learning.

2. **Asisten Virtual (Siri, Google Assistant, ChatGPT)** — Menggunakan Natural
   Language Processing (NLP) untuk memahami perintah suara/teks dan merespons
   dengan informasi atau tindakan yang sesuai.

## Soal 2b: Perbedaan Supervised Learning dan Unsupervised Learning

| Aspek | Supervised Learning | Unsupervised Learning |
|-------|--------------------|-----------------------|
| **Data** | Berlabel (labeled) | Tidak berlabel (unlabeled) |
| **Tujuan** | Memprediksi label/output berdasarkan input | Menemukan struktur/kelompok dalam data |
| **Feedback** | Ada (dari ground truth label) | Tidak ada |
| **Contoh Algoritma** | Linear Regression, Random Forest, SVM | K-Means, DBSCAN, PCA |

**Contoh Supervised Learning:** Klasifikasi email spam — model dilatih dengan
ribuan email berlabel "spam" / "not spam" untuk memprediksi email baru.

**Contoh Unsupervised Learning:** Segmentasi pelanggan — model mengelompokkan
pelanggan ke cluster berdasarkan pola pembelian tanpa label sebelumnya.

## Soal 3a: Apa itu Feature dalam Machine Learning?

**Feature** adalah variabel atau atribut yang dapat diukur/diamati dari data dan
digunakan sebagai input untuk model machine learning. Feature membantu model
memahami pola dan membuat prediksi.

**Contoh:** Prediksi harga rumah -> feature: luas tanah, jumlah kamar, lokasi, dll.

### Kenapa pemilihan fitur penting?
1. **Relevansi** — Fitur tidak relevan menambah noise, menurunkan akurasi
2. **Curse of Dimensionality** — Semakin banyak fitur, semakin banyak data
   dibutuhkan untuk hindari overfitting
3. **Interpretability** — Fitur sedikit + bermakna = model lebih mudah dijelaskan
4. **Efisiensi Komputasi** — Fitur tepat = training lebih cepat dan murah
5. **Bias-Variance Tradeoff** — Fitur salah bisa sebabkan under/overfitting

Teknik pemilihan fitur: feature importance, PCA, mutual information, RFE.

## Soal 3b: Apa itu Fine-tuning dalam Machine Learning?

**Fine-tuning** adalah teknik transfer learning di mana model yang sudah
pre-trained pada dataset besar dan umum dilatih ulang secara spesifik pada
dataset yang lebih kecil untuk tugas tertentu.

Proses: ambil model pre-trained -> ganti head/top layer -> train ulang
dengan learning rate kecil (biasanya hanya beberapa epoch).

### Contoh kasus: Deteksi sampah pesisir

YOLO yang pre-trained di COCO (80 kelas umum) di-fine-tune untuk mendeteksi
sampah plastik di pesisir — kelas yang tidak ada di COCO. Karena model sudah
tahu konsep dasar objek (bentuk, tepi, tekstur), fine-tuning hanya butuh
ratusan gambar berlabel — bukan ribuan. Jauh lebih efisien daripada training
dari scratch.

**Manfaat fine-tuning:**
- Lebih cepat konvergensi (epoch lebih sedikit)
- Butuh data lebih sedikit (ribuan vs jutaan)
- Komputasi lebih murah
- Performa lebih baik daripada training dari nol pada dataset kecil